# Part 3 — Classical Machine Learning

**Industrial Predictive Maintenance & Failure Prevention**

This notebook documents the final classical-ML workflow. The reusable training implementation is `src/models/train_classical_ml.py`.


## 1. Shared feature dataset and leakage control

Part 3 loads the team's existing `data/features/ai4i2020_features.csv` exactly as produced by the earlier project work. **No earlier team files are changed.**

The classifier does not use every column in that file. `src/models/train_classical_ml.py` contains an explicit `APPROVED_FEATURE_COLUMNS` whitelist of current-record sensor variables and safe deterministic engineered features. IDs, the target, direct failure-mode labels (`twf`, `hdf`, `pwf`, `osf`, `rnf`), rolling/lag columns, cumulative target history, and pre-created one-hot duplicates are not passed to the models.

This lets Part 3 benefit from the team's feature dataset while isolating the classical-ML experiment from columns that could leak the target or depend on questionable row ordering.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

cv_results = pd.read_csv(PROJECT_ROOT / "reports" / "classical_ml_cv_results.csv")
test_results = pd.read_csv(PROJECT_ROOT / "reports" / "classical_ml_metrics.csv")
tuning_results = pd.read_csv(PROJECT_ROOT / "reports" / "classical_ml_tuning_results.csv")
threshold_results = pd.read_csv(PROJECT_ROOT / "reports" / "classical_ml_threshold_search.csv")
feature_importance = pd.read_csv(PROJECT_ROOT / "reports" / "classical_ml_feature_importance.csv")
metadata = json.loads((PROJECT_ROOT / "models" / "classical_ml_metadata.json").read_text())

metadata


## 2. Experimental design

- **Target:** `machine_failure`
- **Class imbalance:** about 3.4% failures
- **Train/test split:** stratified 80/20 with random seed 42
- **Cross-validation:** 3-fold stratified CV on the training split
- **Models:** Logistic Regression, Decision Tree, Random Forest, Hist Gradient Boosting
- **Primary tuning metric:** average precision (PR-focused metric)
- **Cost assumption:** false negative = 10 units, false positive = 1 unit
- **Threshold selection:** from out-of-fold training probabilities only
- **Deployment-model selection:** from training/CV business cost only; test results do not choose the model

The 10:1 ratio is a project assumption for demonstrating cost-sensitive decision making, not a measured industrial maintenance cost.


## 3. Cross-validated model selection


In [ ]:
display(
    cv_results.style.format({
        "cv_pr_auc": "{:.4f}",
        "cv_roc_auc": "{:.4f}",
        "cost_threshold": "{:.2f}",
        "oof_precision": "{:.3f}",
        "oof_recall": "{:.3f}",
        "oof_f1": "{:.3f}",
    })
)


The deployment model is the first row of this table because model selection is based on **out-of-fold training business cost**, with CV PR-AUC and recall as tie-breakers. This is deliberately decided before looking at held-out test performance.


## 4. Hyperparameter tuning


In [ ]:
display(tuning_results.head(10))


Only the strongest baseline model by CV PR-focused performance is tuned. If the sampled search fails to improve CV average precision, the stronger baseline configuration is retained instead of forcing a worse tuned model.


## 5. Cost-sensitive threshold selection


In [ ]:
selected_model = metadata["selected_model"]
selected_threshold = metadata["decision_threshold"]

selected_threshold_rows = threshold_results[
    threshold_results["model"].eq(selected_model)
].sort_values("business_cost")

display(selected_threshold_rows.head(10))
print("Selected model:", selected_model)
print("Selected threshold:", selected_threshold)


The threshold is not assumed to be 0.50. Thresholds from 0.01 to 0.99 are evaluated with:

**Cost = 10 × False Negatives + 1 × False Positives**

This prioritizes missed failures more heavily than false alarms while keeping the assumption explicit.


## 6. Held-out test results


In [ ]:
display(test_results)


The test table is for final evaluation and comparison only. A different model may look better on one held-out metric than the deployment-selected model; that does not trigger reselection because doing so would leak test information into model choice.


In [ ]:
display(Image(filename=str(PROJECT_ROOT / "reports" / "figures" / "classical_ml_pr_curves.png")))
display(Image(filename=str(PROJECT_ROOT / "reports" / "figures" / "classical_ml_roc_curves.png")))


In [ ]:
display(Image(filename=str(PROJECT_ROOT / "reports" / "figures" / "classical_ml_threshold_cost.png")))
display(Image(filename=str(PROJECT_ROOT / "reports" / "figures" / "classical_ml_best_confusion_matrix.png")))


## 7. Feature importance


In [ ]:
display(feature_importance.head(12))
display(Image(filename=str(PROJECT_ROOT / "reports" / "figures" / "classical_ml_feature_importance.png")))


Permutation importance is calculated for the selected full pipeline using held-out data. It is an explanatory report, not a feature-selection step, so it does not alter the already-selected model.


## 8. Reproduce Part 3

From the project root:

```bash
pip install -r requirements-ml.txt
python src/models/train_classical_ml.py
```

The training script reads the existing shared feature CSV and regenerates only the Part 3 artifacts: the saved classical-ML pipeline, metadata, metrics, thresholds, predictions, feature importance, and figures.
